In [140]:
from export import build_drivers

drivers = build_drivers(pa, pg, pc)
drivers.to_csv(out_dir / "drivers.csv", index=False)

d = pd.read_csv(out_dir / "drivers.csv")
print(d.shape)
print(d.groupby("race_id")[["p_base", "p_grid", "p_grid_pole"]].sum().round(6).drop_duplicates())
print(d["won"].sum(), d["race_id"].nunique())

(6845, 12)
         p_base  p_grid  p_grid_pole
race_id                             
821         1.0     1.0          1.0
329 329


In [139]:
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

races.to_csv(out_dir / "races.csv", index=False)

check = pd.read_csv(out_dir / "races.csv")
print(check.shape, check["pole_won"].sum(), check["qual_pole_won"].sum(), round(check["gain"].mean(), 4))
check.head(3)

(329, 20) 172 173 0.4411


,race_id,year,round,date,circuit_id,is_sprint_grid,loss_base,loss_grid,gain,pole_driver,pole_won,pole_p_base,pole_margin_ms,qual_pole_won,dominant_season,circuit_name,country,latitude,longitude,circuit_type
0,821,2010,1,2010-03-14,bahrain,False,2.450436,2.566611,-0.116175,sebastian-vettel,0,0.086375,141.0,0,False,Bahrain,bahrain,26.032500,50.510556,RACE
1,822,2010,2,2010-03-28,melbourne,False,2.979442,3.219046,-0.239604,sebastian-vettel,0,0.104390,116.0,0,False,Melbourne,australia,-37.849722,144.968333,STREET
2,823,2010,3,2010-04-04,sepang,False,2.630304,2.485605,0.144699,mark-webber,0,0.070914,1346.0,0,False,Sepang,malaysia,2.760556,101.737500,RACE


In [138]:
races = build_races(dp2, pa, pg, circuits, pm, dominant)
print(races.shape, races["pole_won"].sum(), races["qual_pole_won"].sum())

(329, 20) 172 173


In [137]:
print(races.shape)
print("qual_pole_won" in races.columns)

(329, 19)
False


In [136]:
print(races.shape, races["pole_won"].sum(), races["qual_pole_won"].sum())

KeyError: 'qual_pole_won'

In [134]:
races = build_races(dp2, pa, pg, circuits, pm, dominant)
print(races.shape, races.columns[0])

(329, 19) race_id


In [135]:
q1 = data[data["qualification_position_number"] == 1].groupby("race_id")["won"].max()
g1 = data[data["grid"] == 1].groupby("race_id")["won"].max()

diff = pd.concat([q1.rename("qual_pole_won"), g1.rename("grid_pole_won")], axis=1).fillna(0)
diff = diff[diff["qual_pole_won"] != diff["grid_pole_won"]]

diff.join(races.set_index("race_id")[["year", "round", "circuit_name"]], how="inner")

,qual_pole_won,grid_pole_won,year,round,circuit_name
race_id,,,,,
864,0,1,2012,6,Monaco
1051,0,1,2021,16,Istanbul Park
1071,1,0,2022,14,Spa-Francorchamps
1091,1,0,2023,12,Spa-Francorchamps
1124,1,0,2024,23,Lusail


In [132]:
from export import build_races

circuits = pd.read_sql("SELECT * FROM circuit", con)
races = build_races(dp2, pa, pg, circuits, pm, dominant)

print(races.shape)
print(races["gain"].mean(), races["pole_won"].mean())
print(races[["latitude", "longitude"]].isna().sum())
print(races["pole_margin_ms"].isna().sum(), races["circuit_name"].isna().sum())

(329, 19)
0.4411434691660907 0.5227963525835866
latitude     0
longitude    0
dtype: int64
1 0


In [131]:
bp = pa[pa["grid"] == 1]
print(bp["p"].mean(), bp["won"].mean())

0.22183760458892546 0.5276073619631901


In [130]:
pd.read_sql("SELECT * FROM circuit LIMIT 3", con)

,id,name,full_name,previous_names,type,direction,place_name,country_id,latitude,longitude,length,turns,total_races_held
0,adelaide,Adelaide,Adelaide Street Circuit,NaN,STREET,CLOCKWISE,Adelaide,australia,-34.927222,138.617222,3.780,16,11
1,aida,Aida,Okayama International Circuit,TI Circuit Aida,RACE,CLOCKWISE,Aida,japan,34.915000,134.221111,3.703,13,2
2,ain-diab,Ain-Diab,Ain-Diab Circuit,NaN,ROAD,CLOCKWISE,Casablanca,morocco,33.578611,-7.687500,7.618,18,1


## Summary of results so far

| Question | Result |
|---|---|
| Baseline loss (form + driver) | 1.747 |
| With grid | 1.306 (gain 0.441, 95% range 0.326 to 0.554) |
| With pole flag (extra, not in the original method) | 1.293 (gain 0.454) |
| Against oracle baseline | gain 0.359 |
| Against Friday practice pace | gain 0.384 |
| Non-dominant vs dominant seasons | 0.530 vs 0.347 |
| Eras 2010-13 / 2014-21 / 2022-25 | 0.690 / 0.352 / 0.388 |
| Circuit differences | none reliable (p = 0.42) |
| Pole margin tight / middle / wide | pole wins 48% / 53% / 56% |
| Near-tie front car finishes ahead | 53.5% (51.3% to 55.7%) |
| Penalty-flipped, started-ahead car finishes ahead | 60.2% (51.0% to 69.4%) |

Pole win rate: 52.6% of 329 races (173 wins).

## Slot vs pace 2: grid-penalty pairs

Question: when a penalty or pit-lane start put the faster qualifier behind a slower one on the grid, who finished ahead?

Pairs: neighbouring qualifiers within 0.1 s whose grid order was flipped (108 pairs where both cars finished).

Result: the car that started ahead finished ahead in 60.2% of pairs (approx. 95% range 51.0% to 69.4%). The median grid gap in these pairs is 4 places, much bigger than in the near-tie pairs.

Combined with the near-tie result (53.5%), both checks lean towards some starting-slot advantage, but neither is strong.

Caveats:
- 108 pairs is a small sample, and the range only just clears 50%.
- Penalised cars are not a random sample: a penalty can come from a mechanical change, and some teams take penalties on purpose. Either could make the penalised car finish worse for reasons other than its grid slot.
- The grid gap is larger here than in the near-tie pairs, so the two rates are not the same size of effect.
- Quasi-experimental, not causal.

In [129]:
flip = both[both["grid"] > both["grid_back"]].copy()
n = len(flip)

started_ahead_won = flip["position_number_back"] < flip["position_number"]
p = started_ahead_won.mean()

print(n)
print(p, 1.96 * np.sqrt(p * (1 - p) / n))
print((flip["grid"] - flip["grid_back"]).describe())

108
0.6018518518518519 0.0923233052603898
count    108.000000
mean       4.740741
std        3.109739
min        1.000000
25%        3.000000
50%        4.000000
75%        5.250000
max       17.000000
dtype: float64


## Slot vs pace 1: near-tie qualifying pairs

Question: when two neighbouring cars qualify almost level, does the one in front still finish ahead?

Method: neighbouring pairs whose positions were decided in the same qualifying session, gap measured in that session's lap times (6,966 pairs in 393 races; the 6 sprint-grid races are excluded). 27 pairs dropped (16 with no lap time, 11 where the lower-ranked car was quicker, e.g. after rule decisions). Near-tie = gap under 0.1 s (2,678 pairs). Only pairs where both cars finished are scored (1,943).

Result: the front car finished ahead in 53.5% of pairs (approx. 95% range 51.3% to 55.7%). An earlier build found about 55%.

Caveats:
- A 0.1 s gap is still a small real pace difference, so this is suggestive of a starting-slot effect, not proof, and not a causal claim.
- The range treats pairs as independent, but each car appears in two pairs and races share conditions, so the true range is wider.
- In 5.6% of pairs a penalty put the faster qualifier behind on the grid.
- Pairs with a retirement are excluded, which may bias the rate in a direction I have not tested.

In [128]:
pairs = pairs[pairs["gap_ms"] >= 0].copy()
print(len(pairs))

res = table.set_index(["race_id", "driver_id"])[["position_number", "grid"]]

near = pairs[pairs["gap_ms"] < 100].copy()
near = near.join(res, on=["race_id", "driver_id"])
near = near.join(res.add_suffix("_back"), on=["race_id", "driver_id_back"])

both = near.dropna(subset=["position_number", "position_number_back"])
ahead = both["position_number"] < both["position_number_back"]

n, p = len(both), ahead.mean()
print(len(near), n)
print(p, 1.96 * np.sqrt(p * (1 - p) / n))
print((both["grid"] < both["grid_back"]).mean())

6939
2678 1943
0.534740092640247 0.022178841612808576
0.9444158517756047


In [127]:
bad = pairs[(pairs["gap_ms"] < 0) | pairs["gap_ms"].isna()]

bad[["year", "race_id", "qpos", "driver_id", "driver_id_back",
     "group", "t", "t_back", "gap_ms"]].sort_values(["year", "race_id", "qpos"])

,year,race_id,qpos,driver_id,driver_id_back,group,t,t_back,gap_ms
20,2006,751,21.0,yuji-ide,kimi-raikkonen,1,100270.0,NaN,NaN
130,2006,756,21.0,franck-montagny,david-coulthard,1,80763.0,NaN,NaN
151,2006,757,20.0,franck-montagny,felipe-massa,1,77502.0,NaN,NaN
174,2006,758,21.0,franck-montagny,jarno-trulli,1,86316.0,NaN,NaN
262,2006,762,21.0,sakon-yamamoto,scott-speed,1,80444.0,NaN,NaN
372,2006,767,21.0,tiago-monteiro,sakon-yamamoto,1,93709.0,NaN,NaN
394,2006,768,21.0,sakon-yamamoto,tiago-monteiro,1,73357.0,NaN,NaN
482,2007,772,21.0,christijan-albers,scott-speed,1,83990.0,NaN,NaN
504,2007,773,21.0,takuma-sato,christijan-albers,1,78554.0,NaN,NaN
784,2008,786,15.0,mark-webber,kimi-raikkonen,1,86914.0,86140.0,-774.0


In [126]:
print(pairs["race_id"].nunique(), len(pairs))
print((pairs["gap_ms"] < 0).sum(), pairs["gap_ms"].isna().sum())
print((pairs["gap_ms"] < 100).sum())

p = pairs[pairs["race_id"] == 1102]
print(len(p), p.loc[p["qpos"] == 5, "gap_ms"].iloc[0])

393 6966
11 16
2689
17 5.0


In [125]:
q = pd.read_sql(Path("../sql/04_qualifying_times.sql").read_text(), con)

q["group"] = np.where(q["q3_millis"].notna(), 3,
                      np.where(q["q2_millis"].notna(), 2, 1))
q["t"] = q["q3_millis"].fillna(q["q2_millis"]).fillna(q["q1_millis"])

q = q.sort_values(["race_id", "qpos"])
back = q.groupby("race_id")[["qpos", "group", "t", "driver_id"]].shift(-1)
pairs = q[["race_id", "year", "driver_id", "qpos", "group", "t"]].join(
    back.add_suffix("_back")
)

ok = (pairs["group"] == pairs["group_back"]) & (pairs["qpos_back"] == pairs["qpos"] + 1)
pairs = pairs[ok].copy()
pairs["gap_ms"] = pairs["t_back"] - pairs["t"]

In [123]:
pd.read_sql("""
SELECT r.year, qr.position_number AS pos, qr.driver_id,
       qr.q1_millis, qr.q2_millis, qr.q3_millis,
       qr.time, qr.interval_millis
FROM qualifying_result qr
JOIN race r ON r.id = qr.race_id
WHERE r.year = 2024 AND r.round = 1
ORDER BY qr.position_number
""", con)

,year,pos,driver_id,q1_millis,q2_millis,q3_millis,time,interval_millis
0,2024,1,max-verstappen,90031,89374.0,89179.0,None,NaN
1,2024,2,charles-leclerc,90243,89165.0,89407.0,None,228.0
2,2024,3,george-russell,90350,89922.0,89485.0,None,78.0
3,2024,4,carlos-sainz-jr,89909,89573.0,89507.0,None,22.0
4,2024,5,sergio-perez,90221,89932.0,89537.0,None,30.0
5,2024,6,fernando-alonso,90179,89801.0,89542.0,None,5.0
6,2024,7,lando-norris,90143,89941.0,89614.0,None,72.0
7,2024,8,oscar-piastri,90531,90122.0,89683.0,None,69.0
8,2024,9,lewis-hamilton,90451,89718.0,89710.0,None,27.0
9,2024,10,nico-hulkenberg,90566,89851.0,90502.0,None,792.0


## Robustness 6: uncertainty on the headline gain

Headline: the grid reduces walk-forward loss by 0.441 nats per race (1.747 to 1.306).

Bootstrap, 5,000 resamples, 95% range:

| Resampling unit | Range |
|---|---|
| Individual races (329) | 0.326 to 0.554 |
| Whole seasons (16) | 0.337 to 0.554 |

Result: the gain stays clearly above zero in both. Quote the race-level range. The season-level range is similar, so linkage between races in a season does not change the picture.

Caveat: the season-level resample draws from only 16 units, so it is a rough check.

In [122]:
years = rg["year"].unique()
by_year = {y: rg.loc[rg["year"] == y, "gain"].values for y in years}

boots_y = []
for _ in range(5000):
    pick = rng.choice(years, size=len(years), replace=True)
    boots_y.append(np.concatenate([by_year[y] for y in pick]).mean())

print(np.percentile(boots_y, [2.5, 97.5]))

[0.33725181 0.55353581]


In [121]:
rng = np.random.default_rng(0)
g = rg["gain"].values

boots = [rng.choice(g, size=len(g), replace=True).mean() for _ in range(5000)]

print(g.mean())
print(np.percentile(boots, [2.5, 97.5]))

0.4411434691660907
[0.32619614 0.55356564]


## Robustness 5: circuits and pole margin

### Circuits
Question: does the grid matter more at some tracks than others?

Gain per race (baseline loss minus grid-model loss), averaged by circuit, for the 20 circuits with at least 8 test races. Monaco (+1.02) and Marina Bay (+0.85) were highest, Sochi (-0.20) lowest.

Permutation test (circuit labels shuffled 2,000 times): p = 0.42. A spread this large is common by chance, so there is no reliable evidence that circuits differ. Earlier work found the same (p = 0.68).

Caveat: each circuit has only 8 to 17 races, so circuit averages are noisy.

### Pole margin
Pole margin = qualifying gap between P1 and P2 (median 0.19 s). One race (2015) has no margin and is excluded.

| Margin | Races | Typical margin | Pole wins | Grid gain |
|---|---|---|---|---|
| Tight | 110 | 0.055 s | 48.2% | 0.560 |
| Middle | 109 | 0.186 s | 53.2% | 0.420 |
| Wide | 109 | 0.411 s | 56.0% | 0.347 |

Result: pole wins more often with a wider margin, but the gap (about 8 points) is within the range luck could produce at this sample size. The grid adds information in all three bands, and most in the tight band.

Caveat: tight margins may mostly occur in close seasons, where the grid helps more anyway. This is untested.

In [120]:
m2 = m.dropna(subset=["pole_margin_ms"]).copy()
m2["band"] = pd.qcut(m2["pole_margin_ms"], 3, labels=["tight", "middle", "wide"])

pole_won = (
    pb[pb["grid"] == 1].groupby("race_id")["won"].max().rename("pole_won")
)
m2 = m2.join(pole_won, how="left")
m2["pole_won"] = m2["pole_won"].fillna(0)

m2.groupby("band", observed=True).agg(
    races=("gain", "size"),
    margin_ms=("pole_margin_ms", "median"),
    pole_win_rate=("pole_won", "mean"),
    grid_gain=("gain", "mean"),
)

,races,margin_ms,pole_win_rate,grid_gain
band,,,,
tight,110,55.0,0.481818,0.560372
middle,109,186.0,0.532110,0.419998
wide,109,411.0,0.559633,0.346906


In [119]:
pm = pd.read_sql(Path("../sql/03_pole_margin.sql").read_text(), con)
m = rg.join(pm.set_index("race_id"), how="left")

print(len(m), m["pole_margin_ms"].isna().sum())
print(m.groupby("year")["pole_margin_ms"].apply(lambda s: s.isna().sum()).loc[lambda s: s > 0])
m["pole_margin_ms"].describe()

329 1
year
2015    1
Name: pole_margin_ms, dtype: int64


count     328.000000
mean      240.807927
std       232.506111
min         0.000000
25%        76.750000
50%       185.500000
75%       319.250000
max      1620.000000
Name: pole_margin_ms, dtype: float64

In [118]:
pd.read_sql("SELECT * FROM qualifying_result LIMIT 3", con)

,race_id,position_display_order,position_number,position_text,driver_number,driver_id,constructor_id,engine_manufacturer_id,tyre_manufacturer_id,time,...,q1_millis,q2,q2_millis,q3,q3_millis,gap,gap_millis,interval,interval_millis,laps
0,1,1,1,1,2,nino-farina,alfa-romeo,alfa-romeo,pirelli,1:50.800,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
1,1,2,2,2,3,luigi-fagioli,alfa-romeo,alfa-romeo,pirelli,1:51.000,...,None,None,None,None,None,+0.200,200.0,+0.200,200.0,None
2,1,3,3,3,1,juan-manuel-fangio,alfa-romeo,alfa-romeo,pirelli,1:51.000,...,None,None,None,None,None,+0.200,200.0,+0.000,0.0,None


In [117]:
big = rg[rg.groupby("circuit_id")["gain"].transform("count") >= 8].copy()

def spread(df):
    return df.groupby("circuit_id")["gain"].mean().std()

real = spread(big)

rng = np.random.default_rng(0)
shuffled = []
for _ in range(2000):
    fake = big.copy()
    fake["circuit_id"] = rng.permutation(fake["circuit_id"].values)
    shuffled.append(spread(fake))

print(real)
print(np.mean(np.array(shuffled) >= real))

0.29077754506541387
0.423


In [116]:
pa = walk_forward_probs(data, ["team_form", "driver_rel"])
pg = walk_forward_probs(data, ["team_form", "driver_rel", "log_grid"])

def race_loss(p):
    w = p[p["won"] == 1]
    return pd.Series(-np.log(w["p"].values), index=w["race_id"].values)

gain = (race_loss(pa) - race_loss(pg)).rename("gain")
info = data.drop_duplicates("race_id").set_index("race_id")[["year", "circuit_id"]]
rg = info.join(gain, how="inner")

print(len(rg), rg["gain"].mean())

c = rg.groupby("circuit_id")["gain"].agg(["count", "mean"])
c[c["count"] >= 8].sort_values("mean")

329 0.4411434691660907


,count,mean
circuit_id,,
sochi,8,-0.201930
baku,9,-0.029967
hungaroring,16,0.082588
silverstone,17,0.203702
spa-francorchamps,16,0.246325
melbourne,14,0.292029
bahrain,16,0.409786
interlagos,15,0.425668
austin,13,0.436558


## Robustness 4: era splits

Question: does the grid tell us as much in every era of F1?

Eras (by major car-rule changes): 2010-13, 2014-21, 2022-25.

| Era | Races | Baseline | With grid | Gain |
|---|---|---|---|---|
| 2010-13 | 77 | 2.106 | 1.416 | 0.690 |
| 2014-21 | 160 | 1.687 | 1.334 | 0.352 |
| 2022-25 | 92 | 1.552 | 1.164 | 0.388 |

Result: the grid adds information in every era. The later two eras are close (about 0.35 to 0.39).

Caveat: 2010-13 is scored by models trained on very little history (2010 uses only 2006-09), so its weak baseline probably inflates the gain. The 2014-25 comparison is the cleaner one.

In [115]:
g = wf.merge(wf_b, on="year", suffixes=("_base", "_grid"))
g["era"] = pd.cut(g["year"], [2009, 2013, 2021, 2025],
                  labels=["2010-13", "2014-21", "2022-25"])

out = g.groupby("era", observed=True).agg(
    races=("races_base", "sum"),
    base=("nll_base", "sum"),
    grid=("nll_grid", "sum"),
)
out["base"] = out["base"] / out["races"]
out["grid"] = out["grid"] / out["races"]
out["gain"] = out["base"] - out["grid"]
out

,races,base,grid,gain
era,,,,
2010-13,77,2.105774,1.415897,0.689877
2014-21,160,1.686510,1.334406,0.352104
2022-25,92,1.551575,1.163760,0.387815


## Robustness 3: Friday practice pace

Question: does qualifying still add information once the model also knows how quick each car looked in practice?

Pace feature: each team's quicker driver, best lap across FP1 and FP2, as a % gap to the fastest lap of the weekend. On sprint-shootout weekends (6 per year from 2023) only FP1 exists. 30 missing values (0.4%) filled with the race median.

| Model | Walk-forward loss |
|---|---|
| Form + driver + practice pace | 1.700 |
| Same + grid | 1.316 |
| Gain from the grid | 0.384 |

Result: practice pace adds only 0.05 over the baseline, and the grid keeps almost all of its gain (0.44 without pace, 0.38 with).

Caveats:
- Teams run practice with different fuel loads, engine modes and programmes, so Friday pace is a noisy proxy for qualifying pace. Qualifying adding value beyond it is partly a measure of that noise.
- Earlier work found a smaller gain (+0.29). The practice-pace definition differs, and I have not reconciled the two.
- In 17 races the fastest practice driver is not in the race table (2006 test drivers, and 2 dropped pole-sitters), so the benchmark for the gap is off by a constant there. This should not change within-race probabilities.

In [114]:
dp2 = prepare(dp)
print(dp2["team_pace"].isna().sum())
dp2["team_pace"] = dp2["team_pace"] / 100

wf_p = walk_forward(dp2, cols_p)
wf_pg = walk_forward(dp2, cols_p + ["log_grid"])

lp = wf_p["nll"].sum() / wf_p["races"].sum()
lpg = wf_pg["nll"].sum() / wf_pg["races"].sum()
print(lp, lpg, lp - lpg)

0
1.6995166935159134 1.315578476801277 0.38393821671463635


In [113]:
import warnings
warnings.simplefilter("default")

Z = train[cols_all]
Z = (Z - Z.mean()) / Z.std()

with warnings.catch_warnings():
    warnings.simplefilter("error")
    bz = fit(Z, train["won"], train["race_id"])

raw = fit(train[cols_all], train["won"], train["race_id"])

print(neg_log_lik(bz, Z, train["won"], train["race_id"]))
print(neg_log_lik(raw, train[cols_all], train["won"], train["race_id"]))

RuntimeWarning: divide by zero encountered in log

In [112]:
import warnings
warnings.simplefilter("error")

chk = prepare(dp)
fit(train[cols_all], train["won"], train["race_id"])

warnings.simplefilter("default")

RuntimeWarning: divide by zero encountered in log

In [111]:
view = wf_pg.assign(per_race=wf_pg["nll"] / wf_pg["races"]).round(3)
print(view)

train = dp2[dp2["year"] < 2025]
cols_all = cols_p + ["log_grid"]
print(cols_all)
print(fit(train[cols_all], train["won"], train["race_id"]))

    year  races     nll  per_race
0   2010     19  25.954     1.366
1   2011     19  30.538     1.607
2   2012     20  34.286     1.714
3   2013     19  20.899     1.100
4   2014     19  28.450     1.497
5   2015     19  17.868     0.940
6   2016     21  22.595     1.076
7   2017     20  27.474     1.374
8   2018     21  31.845     1.516
9   2019     21  27.909     1.329
10  2020     17  22.307     1.312
11  2021     22  31.788     1.445
12  2022     22  32.215     1.464
13  2023     22  17.615     0.801
14  2024     24  37.005     1.542
15  2025     24  22.743     0.948
['team_form', 'driver_rel', 'team_pace', 'log_grid']
[  4.95180386   3.76693377 -43.97716653  -1.37969925]


/Users/oj/Documents/GitHub/f1-qualifying-analysis/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [110]:
dp2 = prepare(dp)
dp2["team_pace"] = dp2["team_pace"] / 100

cols_p = ["team_form", "driver_rel", "team_pace"]
wf_p = walk_forward(dp2, cols_p)
wf_pg = walk_forward(dp2, cols_p + ["log_grid"])

lp = wf_p["nll"].sum() / wf_p["races"].sum()
lpg = wf_pg["nll"].sum() / wf_pg["races"].sum()
print(lp, lpg, lp - lpg)

/Users/oj/Documents/GitHub/f1-qualifying-analysis/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/oj/Documents/GitHub/f1-qualifying-analysis/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/oj/Documents/GitHub/f1-qualifying-analysis/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


1.6970545862926147 1.3115223969596121 0.3855321893330026


In [109]:
bad = r[r["best"].ne(0) & r["best"].notna()].index

fast = practice[practice["race_id"].isin(bad)].dropna(subset=["time_millis"])
fast = fast.loc[fast.groupby("race_id")["time_millis"].idxmin()]

racers = set(zip(data["race_id"], data["driver_id"]))
fast["in_race_table"] = [(a, b) in racers for a, b in zip(fast["race_id"], fast["driver_id"])]

print(fast["in_race_table"].value_counts())
fast[~fast["in_race_table"]][["race_id", "driver_id"]].head(20)

in_race_table
False    17
Name: count, dtype: int64


,race_id,driver_id
23535,751,anthony-davidson
7444,752,alexander-wurz
23591,753,anthony-davidson
7518,755,alexander-wurz
23703,757,alexander-wurz
23731,758,robert-kubica
7623,759,robert-kubica
23789,760,anthony-davidson
7674,761,robert-kubica
23847,762,robert-kubica


In [108]:
r = dp.groupby("race_id").agg(
    year=("year", "first"),
    round=("round", "first"),
    best=("team_pace", "min"),
)
r[r["best"].ne(0)].sort_values(["year", "round"])

,year,round,best
race_id,,,
751,2006,1,0.435673
752,2006,2,0.905778
753,2006,3,0.450347
755,2006,5,0.846013
757,2006,7,0.304320
758,2006,8,1.494783
759,2006,9,0.922896
760,2006,10,0.617944
761,2006,11,0.916738


In [107]:
from clean import add_practice_pace

practice = pd.read_sql(Path("../sql/02_practice_times.sql").read_text(), con)
dp = add_practice_pace(data, practice)
d1 = dp[dp["year"].between(2006, 2025)]

print(len(dp) == len(data))
print(d1["team_pace"].isna().sum(), len(d1))
print(d1.loc[d1["team_pace"].isna(), "year"].value_counts().sort_index())
print(dp.groupby("race_id")["team_pace"].min().eq(0).mean())

True
30 8313
year
2006     2
2014     2
2017     4
2020    20
2023     2
Name: count, dtype: int64
0.9565217391304348


In [106]:
pd.read_sql("""
SELECT r.year,
       COUNT(DISTINCT r.id) AS races,
       COUNT(DISTINCT fp1.race_id) AS fp1,
       COUNT(DISTINCT fp2.race_id) AS fp2,
       COUNT(DISTINCT fp3.race_id) AS fp3
FROM race r
LEFT JOIN free_practice_1_result fp1 ON fp1.race_id = r.id
LEFT JOIN free_practice_2_result fp2 ON fp2.race_id = r.id
LEFT JOIN free_practice_3_result fp3 ON fp3.race_id = r.id
WHERE r.year BETWEEN 2006 AND 2025
GROUP BY r.year
ORDER BY r.year
""", con)

,year,races,fp1,fp2,fp3
0,2006,18,18,18,18
1,2007,17,17,17,17
2,2008,18,18,18,18
3,2009,17,17,17,17
4,2010,19,19,19,19
5,2011,19,19,19,19
6,2012,20,20,20,20
7,2013,19,19,19,19
8,2014,19,19,19,19
9,2015,19,19,18,19


In [105]:
pd.read_sql("SELECT * FROM free_practice_2_result LIMIT 5", con)

,race_id,position_display_order,position_number,position_text,driver_number,driver_id,constructor_id,engine_manufacturer_id,tyre_manufacturer_id,time,time_millis,gap,gap_millis,interval,interval_millis,laps
0,435,1,1,1,12,ayrton-senna,lotus,renault,goodyear,1:17.977,77977,NaN,NaN,NaN,NaN,None
1,435,2,2,2,20,gerhard-berger,benetton,bmw,pirelli,1:18.088,78088,+0.111,111.0,+0.111,111.0,None
2,435,3,3,3,19,teo-fabi,benetton,bmw,pirelli,1:18.154,78154,+0.177,177.0,+0.066,66.0,None
3,435,4,4,4,6,nelson-piquet,williams,honda,goodyear,1:18.353,78353,+0.376,376.0,+0.199,199.0,None
4,435,5,5,5,5,nigel-mansell,williams,honda,goodyear,1:18.785,78785,+0.808,808.0,+0.432,432.0,None


## Robustness 2: dominant vs non-dominant seasons

Question: does the grid only help when one team runs away with the season?

Split: for each test season (2010-2025), the share of races won by its most successful team. The 8 seasons above the median (0.66) are "dominant", the other 8 "non-dominant". Borderline seasons: 2011 and 2013.

| Seasons | Races | Baseline | With grid | Gain |
|---|---|---|---|---|
| Non-dominant | 169 | 1.945 | 1.415 | 0.530 |
| Dominant | 160 | 1.538 | 1.191 | 0.347 |

Result: the grid adds information in both groups, and more in close seasons (where the car alone leaves more uncertainty).

Note: the split rule is my own definition, so the exact gap between groups is approximate.

In [104]:
dominant = share[share > share.median()].index

g = wf.merge(wf_b, on="year", suffixes=("_base", "_grid"))
g["dominant"] = g["year"].isin(dominant)

out = g.groupby("dominant").agg(
    races=("races_base", "sum"),
    base=("nll_base", "sum"),
    grid=("nll_grid", "sum"),
)
out["base"] = out["base"] / out["races"]
out["grid"] = out["grid"] / out["races"]
out["gain"] = out["base"] - out["grid"]
out

,races,base,grid,gain
dominant,,,,
False,169,1.944995,1.414520,0.530475
True,160,1.537669,1.190882,0.346787


In [103]:
wins = data[data["won"] == 1].groupby(["year", "team"]).size()
top_wins = wins.groupby("year").max()
races = data.groupby("year")["race_id"].nunique()

share = (top_wins / races).loc[2010:2025]
print(share.median())
share.sort_values()

0.6578947368421053


year
2012    0.350000
2024    0.375000
2010    0.473684
2021    0.500000
2018    0.523810
2025    0.583333
2017    0.600000
2011    0.631579
2013    0.684211
2019    0.714286
2020    0.764706
2022    0.772727
2014    0.842105
2015    0.842105
2016    0.904762
2023    0.954545
dtype: float64

## Robustness 1: oracle baseline

Question: is the gain from qualifying just qualifying revealing car strength?

Test: replace past-race team form with each team's full-season average (uses future races, so no real forecaster could have it), then add the grid.

| Model | Walk-forward loss (nats/race) |
|---|---|
| Oracle baseline | 1.608 |
| Oracle + grid | 1.250 |
| Gain from the grid | 0.359 |

Result: the grid still adds 0.36 nats/race on top of full-season car strength (about 5.0 plausible winners narrowing to 3.5).

Note: this version uses `log_grid` only, without the pole flag.

In [102]:
oracle_b = walk_forward(data_o, ["season_form", "driver_rel", "log_grid"])
loss_b = oracle_b["nll"].sum() / oracle_b["races"].sum()
loss_a = oracle_a["nll"].sum() / oracle_a["races"].sum()

print(loss_b)
print(loss_a - loss_b)

1.2496030716220916
0.35868706783590487


In [101]:
from model import add_season_strength

data_o = add_season_strength(data)

oracle_a = walk_forward(data_o, ["season_form", "driver_rel"])
print(oracle_a["nll"].sum() / oracle_a["races"].sum())

1.6082901394579965


In [100]:
pc["bucket"] = pd.cut(pc["p"], [0, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7, 1.0])

pc.groupby("bucket", observed=True).agg(
    n=("p", "size"),
    predicted=("p", "mean"),
    actual=("won", "mean"),
)

,n,predicted,actual
bucket,,,
"(0.0, 0.02]",5303,0.002687,0.003017
"(0.02, 0.05]",476,0.032380,0.029412
"(0.05, 0.1]",303,0.070981,0.066007
"(0.1, 0.2]",187,0.144180,0.176471
"(0.2, 0.35]",213,0.269268,0.262911
"(0.35, 0.5]",133,0.418946,0.375940
"(0.5, 0.7]",207,0.578923,0.579710
"(0.7, 1.0]",23,0.780816,0.869565


In [99]:
data = prepare(table)
cols = ["team_form", "driver_rel", "log_grid", "is_pole"]

wf_c = walk_forward(data, cols)
print(wf_c["nll"].sum() / wf_c["races"].sum())

pc = walk_forward_probs(data, cols)
pole = pc[pc["grid"] == 1]
print(pole["p"].mean(), pole["won"].mean())

1.2932770722774416
0.5391775898895704 0.5276073619631901


In [98]:
for T in [0.8, 0.9, 0.95, 1.0]:
    q = temper(pb["p"], pb["race_id"], T)
    loss = -np.log(q[pb["won"] == 1]).sum() / (pb["won"] == 1).sum()
    print(T, round(loss, 4))

0.8 1.3493
0.9 1.3156
0.95 1.3082
1.0 1.3058


In [97]:
from model import temper

for T in [1.0, 1.1, 1.2, 1.3, 1.4]:
    q = temper(pb["p"], pb["race_id"], T)
    loss = -np.log(q[pb["won"] == 1]).sum() / (pb["won"] == 1).sum()
    print(T, round(loss, 4))

1.0 1.3058
1.1 1.3126
1.2 1.3312
1.3 1.358
1.4 1.3902


In [96]:
pb["bucket"] = pd.cut(pb["p"], [0, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7, 1.0])

pb.groupby("bucket", observed=True).agg(
    n=("p", "size"),
    predicted=("p", "mean"),
    actual=("won", "mean"),
)

,n,predicted,actual
bucket,,,
"(0.0, 0.02]",5200,0.003426,0.001923
"(0.02, 0.05]",589,0.032000,0.032258
"(0.05, 0.1]",351,0.069639,0.068376
"(0.1, 0.2]",263,0.147553,0.231939
"(0.2, 0.35]",112,0.251169,0.321429
"(0.35, 0.5]",59,0.438658,0.423729
"(0.5, 0.7]",204,0.609011,0.514706
"(0.7, 1.0]",67,0.758748,0.731343


In [95]:
from model import walk_forward_probs

pb = walk_forward_probs(data, ["team_form", "driver_rel", "log_grid"])
print(pb.groupby("race_id")["p"].sum().round(6).unique())

pole = pb[pb["grid"] == 1]
print(pole["p"].mean(), pole["won"].mean())

[1.]
0.6068335991823439 0.5276073619631901


In [94]:
data = prepare(table)

wf_b = walk_forward(data, ["team_form", "driver_rel", "log_grid"])
print(wf_b["nll"].sum() / wf_b["races"].sum())
print(wf["nll"].sum() / wf["races"].sum() - wf_b["nll"].sum() / wf_b["races"].sum())

1.3057597151724105
0.4411434691660907


In [93]:
from model import walk_forward

wf = walk_forward(data, ["team_form", "driver_rel"])
print(wf["races"].sum())
print(wf["nll"].sum() / wf["races"].sum())
wf

329
1.7469031843385012


,year,races,nll
0,2010,19,43.882037
1,2011,19,34.805159
2,2012,20,48.488327
3,2013,19,34.969076
4,2014,19,39.274026
5,2015,19,27.368940
6,2016,21,28.982521
7,2017,20,37.068284
8,2018,21,38.723304
9,2019,21,31.181682


In [57]:
from model import fit

beta = fit(X, d["won"], d["race_id"])
print(beta)
print(neg_log_lik(beta, X, d["won"], d["race_id"]) / d["race_id"].nunique())

[11.79002262  5.66340953]
1.7111991182539088


In [58]:
import numpy as np

from model import neg_log_lik

d = data[data["year"].between(2010, 2025)]
X = d[["team_form", "driver_rel"]]

nll = neg_log_lik(np.zeros(2), X, d["won"], d["race_id"])
print(nll / d["race_id"].nunique())
print(np.log(d.groupby("race_id").size()).mean())

3.032082440104498
3.032082440104498


In [59]:
from model import prepare

data = prepare(table)
print(data.groupby("race_id")["won"].sum().value_counts())

won
1    414
Name: count, dtype: int64


## Data notes: missing pole-sitters

In 3 of 329 test races (2010-2025) the qualifying pole-sitter has no row in the cleaned table:
- 2021 R5: DNS (never started)
- 2023 R18 and 2025 R22: started from pole, disqualified after the race (DSQ rows are dropped)

Treatment: the headline pole win rate uses all 329 races, counting these as non-wins (173/329 = 52.6%). Robustness check later: refit without these 3 races.

In [60]:
df[
    df["race_id"].isin(missing)
    & ((df["qualification_position_number"] == 1) | df["grid_position_number"].isna())
][["year", "round", "driver_id", "qualification_position_number",
   "grid_position_number", "grid_position_text", "position_text"]]

,year,round,driver_id,qualification_position_number,grid_position_number,grid_position_text,position_text
6214,2021,5,charles-leclerc,1.0,NaN,NaN,DNS
7341,2023,18,lance-stroll,19.0,NaN,PL,7
7345,2023,18,nico-hulkenberg,16.0,NaN,PL,11
7348,2023,18,kevin-magnussen,14.0,NaN,PL,14
7350,2023,18,fernando-alonso,17.0,NaN,PL,DNF
7354,2023,18,charles-leclerc,1.0,1.0,1,DSQ
8345,2025,22,yuki-tsunoda,19.0,NaN,PL,12
8352,2025,22,lando-norris,1.0,1.0,1,DSQ


In [61]:
has_pole = test.groupby("race_id")["grid"].apply(lambda s: (s == 1).any())
missing = has_pole[~has_pole].index

df[df["race_id"].isin(missing) & (df["grid_position_number"] == 1)][
    ["year", "round", "driver_id", "grid_position_number", "position_text"]
]

,year,round,driver_id,grid_position_number,position_text
7354,2023,18,charles-leclerc,1.0,DSQ
8352,2025,22,lando-norris,1.0,DSQ


In [62]:
print((test["grid"] == 1).sum(), (test["qualification_position_number"] == 1).sum())

326 326


In [63]:
test = table[table["year"].between(2010, 2025)]
print(test["race_id"].nunique())
print(test.loc[test["grid"] == 1, "position_number"].eq(1).mean())
print(test.loc[test["qualification_position_number"] == 1, "position_number"].eq(1).mean())

329
0.5276073619631901
0.5306748466257669


In [64]:
from clean import build_table

table = build_table(df, chron)
print(table.shape, table.duplicated(["race_id", "driver_id"]).sum())
print(table[["grid", "team_form", "driver_rel"]].isna().sum())

(8636, 21) 0
grid            0
team_form      32
driver_rel    105
dtype: int64


In [65]:
from clean import add_driver_rel

df6 = add_driver_rel(df5)
print(df6["driver_rel"].isna().sum(), df6["driver_id"].nunique())
df6[df6["driver_id"] == "lewis-hamilton"].head(3)[
    ["year", "round", "teammate_gap", "driver_rel"]
]

105 103


,year,round,teammate_gap,driver_rel
398,2007,1,-0.045455,NaN
419,2007,2,-0.045455,-0.045455
441,2007,3,0.136364,-0.045455


In [66]:
from clean import add_teammate_gap

df5 = add_teammate_gap(df4)
pairs = df5.groupby(["race_id", "constructor_id"])["teammate_gap"]
print(pairs.sum().abs().max())
print(df5["teammate_gap"].isna().sum())
df5[(df5["year"] == 2024) & (df5["round"] == 1) & (df5["constructor_id"] == "red-bull")][
    ["driver_id", "finish_score", "teammate_gap"]
]

2.220446049250313e-16
74


,driver_id,finish_score,teammate_gap
7435,max-verstappen,1.00,0.05
7436,sergio-perez,0.95,-0.05


In [67]:
from clean import add_team_form

teams3 = add_team_form(teams2)
print(teams3["team_form"].isna().sum())
teams3[teams3["team"] == "ferrari"].head(3)[["year", "round", "team_score", "team_form"]]

16


,year,round,team_score,team_form
1,2006,1,0.795455,NaN
12,2006,2,0.795455,0.795455
23,2006,3,0.000000,0.795455


In [68]:
from clean import lineage_labels, add_lineage

labels = lineage_labels(chron)
teams2 = add_lineage(teams, labels)

print(teams2["constructor_id"].nunique(), teams2["team"].nunique())
print(teams2.duplicated(["race_id", "team"]).sum())

34 16
0


In [69]:
latest = (
    chron.sort_values("year_from")
    .groupby("constructor_id")["other_constructor_id"]
    .last()
)

print(set(chron["other_constructor_id"]) - set(chron["constructor_id"]))

labelled = chron.assign(label=chron["constructor_id"].map(latest))
print(labelled.groupby("other_constructor_id")["label"].nunique().max())

latest[["lotus-f1", "lotus-racing", "toro-rosso", "honda"]]

set()
1


constructor_id
lotus-f1              alpine
lotus-racing        caterham
toro-rosso      racing-bulls
honda               mercedes
Name: other_constructor_id, dtype: str

In [70]:
lineage = chron[["other_constructor_id", "constructor_id"]].drop_duplicates()
print(lineage["other_constructor_id"].duplicated().sum())

print(set(teams["constructor_id"]) - set(lineage["other_constructor_id"]))

lineage[lineage["other_constructor_id"].str.startswith("lotus")]

154
{'haas', 'mclaren', 'cadillac', 'toyota', 'hrt', 'ferrari', 'williams', 'super-aguri'}


,other_constructor_id,constructor_id
14,lotus-f1,alpine
42,lotus-f1,benetton
56,lotus-racing,caterham
109,lotus-f1,lotus-f1
112,lotus-racing,lotus-racing
170,lotus-f1,renault
191,lotus-f1,toleman


In [71]:
chron = pd.read_sql("SELECT * FROM constructor_chronology", con)
print(chron.shape)
chron.head(10)

(217, 5)


,constructor_id,position_display_order,other_constructor_id,year_from,year_to
0,alfa-romeo,1,sauber,1993,2005.0
1,alfa-romeo,2,bmw-sauber,2006,2010.0
2,alfa-romeo,3,sauber,2011,2018.0
3,alfa-romeo,4,alfa-romeo,2019,2023.0
4,alfa-romeo,5,kick-sauber,2024,2025.0
5,alfa-romeo,6,audi,2026,NaN
6,alphatauri,1,minardi,1985,2005.0
7,alphatauri,2,toro-rosso,2006,2019.0
8,alphatauri,3,alphatauri,2020,2023.0
9,alphatauri,4,rb,2024,2024.0


In [72]:
span = teams.groupby("constructor_id")["year"].agg(["min", "max", "count"])
span.sort_values("min")

,min,max,count
constructor_id,,,
williams,2006,2026,414
toyota,2006,2009,70
toro-rosso,2006,2019,268
super-aguri,2006,2008,39
bmw-sauber,2006,2009,70
renault,2006,2020,207
ferrari,2006,2026,413
red-bull,2006,2026,414
honda,2006,2008,53


In [73]:
from clean import team_race_scores

teams = team_race_scores(df4)
print(teams.shape)
teams[(teams["year"] == 2024) & (teams["round"] == 1)].head(3)

(4355, 5)


,race_id,year,round,constructor_id,team_score
3713,1102,2024,1,alpine,0.175
3714,1102,2024,1,aston-martin,0.575
3715,1102,2024,1,ferrari,0.875


In [74]:
from clean import add_finish_score

df4 = add_finish_score(df3)
print(df4.loc[df4["finish_score"] == 0, "position_text"].value_counts())
print(df4.groupby("race_id")["finish_score"].max().eq(1.0).all())

position_text
DNF    1412
NC       10
Name: count, dtype: int64
True


In [75]:
from clean import flag_sprint_grid

df3 = flag_sprint_grid(fill_pit_lane_grid(drop_non_starters(df)))
print(df3["grid"].isna().sum())
print(df3.loc[df3["is_sprint_grid"], "race_id"].nunique())

0
6


In [76]:
print(df2["grid"].isna().sum())
print(df2["race_id"].dtype)

df2[
    (df2["driver_id"] == "pierre-gasly")
    & (df2["year"] == 2021)
    & (df2["round"] == 14)
][["driver_id", "grid_position_number", "grid"]]

0
int64


,driver_id,grid_position_number,grid
6393,pierre-gasly,NaN,20.0


In [77]:
print(df2["grid"].isna().sum())

df2[df2["grid_position_number"].isna()][
    ["race_id", "driver_id", "grid_position_number", "grid"]
].head(10)

0


,race_id,driver_id,grid_position_number,grid
16,751,tiago-monteiro,NaN,23.0
48,753,giancarlo-fisichella,NaN,23.0
136,757,michael-schumacher,NaN,22.0
201,760,jarno-trulli,NaN,22.0
260,762,sakon-yamamoto,NaN,21.0
520,774,christijan-albers,NaN,23.0
576,777,felipe-massa,NaN,22.0
585,777,takuma-sato,NaN,23.0
712,783,vitantonio-liuzzi,NaN,23.0
785,786,adrian-sutil,NaN,22.0


In [78]:
from clean import fill_pit_lane_grid

df2 = fill_pit_lane_grid(df1)
print(df2["grid"].isna().sum())
df2[df2["driver_id"].eq("pierre_gasly") & df2["race_id"].eq(1049)][
    ["driver_id", "grid_position_number", "grid"]
]

0


,driver_id,grid_position_number,grid


In [79]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")
from clean import drop_non_starters

df1 = drop_non_starters(df)
len(df), len(df1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(8724, 8636)

## Data notes: sprint-format races

In 2021-22 (6 races, `qualifying_format = SPRINT_RACE`) the Sunday grid was set by the Saturday sprint result, not by qualifying. Check: only 55% of drivers started where they qualified, vs 72% in normal 2021-22 races.

From 2023 the Sunday grid always comes from a qualifying session, so those weekends are unaffected.

Treatment: keep the 6 races and flag them (`is_sprint_grid`). Robustness check later: refit without them. Exclude them from the qualifying-gap and penalty analysis.

In [80]:
pd.read_sql("""
SELECT r.qualifying_format,
    COUNT(*) AS drivers,
    SUM(rr.grid_position_number = rr.qualification_position_number) AS same
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2021 AND 2022
AND rr.grid_position_number IS NOT NULL
AND rr.qualification_position_number IS NOT NULL
GROUP BY r.qualifying_format
""", con)

,qualifying_format,drivers,same
0,KNOCKOUT,733,527
1,SPRINT_RACE,112,62


In [81]:
pd.read_sql("""
SELECT year, qualifying_format, sprint_qualifying_format,
       COUNT(*) AS races
FROM race
WHERE year BETWEEN 2006 AND 2026
GROUP BY year, qualifying_format, sprint_qualifying_format
ORDER BY year
""", con)

,year,qualifying_format,sprint_qualifying_format,races
0,2006,KNOCKOUT,NaN,18
1,2007,KNOCKOUT,NaN,17
2,2008,KNOCKOUT,NaN,18
3,2009,KNOCKOUT,NaN,17
4,2010,KNOCKOUT,NaN,19
5,2011,KNOCKOUT,NaN,19
6,2012,KNOCKOUT,NaN,20
7,2013,KNOCKOUT,NaN,19
8,2014,KNOCKOUT,NaN,19
9,2015,KNOCKOUT,NaN,19


In [82]:
pd.read_sql("""
SELECT year, round, official_name, qualifying_format, sprint_qualifying_format
FROM race
WHERE id = 1049
""", con)

,year,round,official_name,qualifying_format,sprint_qualifying_format
0,2021,14,Formula 1 Heineken Gran Premio d'Italia 2021,SPRINT_RACE,None


In [83]:
no_grid[no_grid["position_text"] == "DNF"]

,race_id,year,round,date,circuit_id,driver_id,constructor_id,qualification_position_number,qualification_position_text,grid_position_number,grid_position_text,position_number,position_text
6393,1049,2021,14,2021-09-12,monza,pierre-gasly,alphatauri,NaN,DNF,NaN,NaN,NaN,DNF


In [84]:
no_grid["position_text"].value_counts()

position_text
DNS    16
DNQ     2
DNP     2
DNF     1
Name: count, dtype: int64

In [85]:
no_grid = df[df["grid_position_text"].isna()]
no_grid["position_number"].notna().value_counts()

position_number
False    21
Name: count, dtype: int64

In [86]:
from pathlib import Path

query = Path("../sql/01_race_results.sql").read_text()
df = pd.read_sql(query, con)
df.shape

(8724, 14)

## Data notes: grid positions

Blank `grid_position_number` values (2006-2025, 191 rows):

| Grid label | Meaning | Treatment |
|---|---|---|
| PL | Pit-lane start | Ranked behind the last gridded car, ordered by qualifying position |
| NaN | No grid information | Check whether the driver finished; drop only if they never started |

Missing qualifying positions (NC, EX, DSQ) don't affect the main model, which uses the grid slot. They matter later for the qualifying-gap and penalty analysis.

In [87]:
query = """
SELECT rr.grid_position_text, rr.qualification_position_text,
    COUNT(*) AS n
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2025
AND (rr.grid_position_number IS NULL
    OR rr.qualification_position_number IS NULL)
GROUP BY rr.grid_position_text, rr.qualification_position_text
ORDER BY n DESC
"""

pd.read_sql(query, con)

,grid_position_text,qualification_position_text,n
0,20,NC,26
1,PL,19,26
2,PL,20,24
3,PL,18,18
4,PL,NC,15
5,19,NC,14
6,22,NC,10
7,PL,14,9
8,PL,16,9
9,PL,10,8


In [88]:
query = """
SELECT r.year,
    COUNT(DISTINCT r.id) AS races,
    COUNT(*) AS rows,
    COUNT(*) - COUNT(rr.grid_position_number) AS no_grid,
    COUNT(*) - COUNT(rr.qualification_position_number) AS no_qual
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2026
GROUP BY r.year
ORDER BY r.year
"""

pd.read_sql(query, con)

,year,races,rows,no_grid,no_qual
0,2006,18,396,5,2
1,2007,17,374,4,0
2,2008,18,368,11,1
3,2009,17,340,9,3
4,2010,19,456,9,1
5,2011,19,454,2,14
6,2012,20,480,6,10
7,2013,19,418,1,5
8,2014,19,407,14,12
9,2015,19,380,7,12


## Data notes: non-finishers

Every non-finish has a blank `position_number`, with the reason in `position_text` (2006-2025 counts):

| Label | Count | Treatment |
|---|---|---|
| DNF | 1,355 | Counts as 0 in team form |
| NC | 8 | Counts as 0 in team form |
| DNS | 49 | Row dropped (never raced) |
| DSQ | 28 | Row dropped (rules breach, not pace) |
| DNQ / DNP | 2 / 2 | Row dropped (never raced) |

Caveat: DNF mixes crashes and mechanical failures, so team form partly reflects reliability, not just speed.

In [89]:
query = """
SELECT rr.position_text, COUNT(*) AS n
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year BETWEEN 2006 AND 2025
AND rr.position_number IS NULL
GROUP BY rr.position_text
ORDER BY n DESC
"""

pd.read_sql(query, con)

,position_text,n
0,DNF,1355
1,DNS,49
2,DSQ,28
3,NC,8
4,DNQ,2
5,DNP,2


In [90]:
query = """
SELECT r.year, r.round, rr.driver_id,
       rr.qualification_position_number AS qual,
       rr.grid_position_number AS grid,
       rr.position_number, rr.position_text, rr.reason_retired
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year = 2024
AND (rr.position_number IS NULL
    OR rr.qualification_position_number != rr.grid_position_number)
ORDER BY r.round
LIMIT 15
"""

pd.read_sql(query, con)

,year,round,driver_id,qual,grid,position_number,position_text,reason_retired
0,2024,2,lance-stroll,10,10,NaN,DNF,Accident
1,2024,2,pierre-gasly,18,18,NaN,DNF,Gearbox
2,2024,3,charles-leclerc,5,4,2.0,2,NaN
3,2024,3,lando-norris,4,3,3.0,3,NaN
4,2024,3,oscar-piastri,6,5,4.0,4,NaN
5,2024,3,sergio-perez,3,6,5.0,5,NaN
6,2024,3,lewis-hamilton,11,11,NaN,DNF,Power unit
7,2024,3,max-verstappen,1,1,NaN,DNF,Brakes
8,2024,4,guanyu-zhou,20,20,NaN,DNF,Gearbox
9,2024,4,daniel-ricciardo,11,11,NaN,DNF,Collision


In [91]:
query = """
SELECT r.year, r.round, rr.driver_id,rr.constructor_id,
       rr.qualification_position_number, rr.grid_position_number,
       rr.position_number, rr.position_text
FROM race_result rr
JOIN race r ON r.id = rr.race_id
WHERE r.year = 2024 AND r.round = 1
ORDER BY rr.position_display_order
"""

pd.read_sql(query, con)

,year,round,driver_id,constructor_id,qualification_position_number,grid_position_number,position_number,position_text
0,2024,1,max-verstappen,red-bull,1,1,1,1
1,2024,1,sergio-perez,red-bull,5,5,2,2
2,2024,1,carlos-sainz-jr,ferrari,4,4,3,3
3,2024,1,charles-leclerc,ferrari,2,2,4,4
4,2024,1,george-russell,mercedes,3,3,5,5
5,2024,1,lando-norris,mclaren,7,7,6,6
6,2024,1,lewis-hamilton,mercedes,9,9,7,7
7,2024,1,oscar-piastri,mclaren,8,8,8,8
8,2024,1,fernando-alonso,aston-martin,6,6,9,9
9,2024,1,lance-stroll,aston-martin,12,12,10,10


In [92]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/raw/f1db.db")